# 📘 Phần 4: Chuẩn Hóa Chuỗi Văn Bản & Xử Lý Ngoại Lai (Text Normalization & Outlier Handling)

Trong các bộ dữ liệu thực tế, hai nhóm lỗi thường xuyên gây suy giảm chất lượng dữ liệu nhất là:
1. **Lỗi chuỗi văn bản (Messy Text)**: Khoảng trắng dư thừa, hoa/thường không đồng nhất, định danh thành phố/địa chỉ viết theo nhiều kiểu khác nhau, định dạng email/sđt không chuẩn.
2. **Giá trị ngoại lai (Outliers / Anomalies)**: Các số liệu bất thường do lỗi nhập liệu (tuổi âm, tuổi > 200, số lượng âm hoặc số lượng lớn bất thường) làm sai lệch các phép toán trung bình và phân phối.

---

## 🎯 Mục Tiêu Bài Học:
1. Chuẩn hóa chuỗi ký tự (`.str.strip()`, `.str.lower()`, `.str.title()`, regex).
2. Ánh xạ và chuẩn hóa tên thành phố / danh mục bằng từ điển (Dictionary Mapping).
3. Kiểm tra tính hợp lệ của định dạng Email bằng Biểu thức chính quy (Regex).
4. Phát hiện giá trị ngoại lai bằng phương pháp **IQR (Interquartile Range)** và **Z-Score**.
5. Các chiến lược xử lý Outlier: **Loại bỏ (Trimming)**, **Giới hạn biên (Capping / Winsorizing với `.clip()`)** và **Chuyển thành NaN để Impute**.


In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Đọc dữ liệu mẫu và khử duplicate cơ bản
df = pd.read_csv("data/customer_orders_raw.csv")
df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
df.head(10)


## 1. Chuẩn Hóa Văn Bản (String Cleaning & Text Normalization)

### A. Xử Lý Khoảng Trắng & Viết Hoa Thường
- Dùng `.str.strip()` để loại bỏ khoảng trắng thừa ở hai đầu chuỗi.
- Dùng `r'\s+'` để thay thế nhiều khoảng trắng liên tiếp bằng một khoảng trắng đơn.
- Chuẩn hóa tên người bằng `.str.title()`.


In [ ]:
# Làm sạch cột customer_name
df['customer_name_clean'] = (
    df['customer_name']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

df[['customer_name', 'customer_name_clean']]


### B. Chuẩn Hóa Tên Thành Phố (Standardizing Inconsistent Categories)
Các tên địa danh thường bị gõ sai hoặc không đồng nhất: `'Ha Noi'`, `'HA NOI'`, `'ha noi '`, `'Hà Nội'`, `'hcm'`, `'Ho Chi Minh'`.


In [ ]:
# 1. Chuyển về chữ thường và xóa khoảng trắng trước khi map
df['city_raw'] = df['city'].astype(str).str.strip().str.lower()

# 2. Xây dựng bảng ánh xạ (Mapping Dictionary)
city_mapping = {
    'ha noi': 'Ha Noi',
    'hà nội': 'Ha Noi',
    'hcm': 'Ho Chi Minh',
    'ho chi minh': 'Ho Chi Minh',
    'da nang': 'Da Nang',
    'hai phong': 'Hai Phong',
    'can tho': 'Can Tho',
    'cần thơ': 'Can Tho',
    'hue': 'Hue',
    'quang ninh': 'Quang Ninh',
    'nha trang': 'Nha Trang',
    '-': np.nan,
    'nan': np.nan
}

df['city_clean'] = df['city_raw'].map(city_mapping)
df[['city', 'city_raw', 'city_clean']]


### C. Kiểm Tra Tính Hợp Lệ Của Email Bằng Regex
Kiểm tra xem email có đúng định dạng chuẩn (`username@domain.ext`) hay không.


In [ ]:
# Regex chuẩn cho định dạng email
email_regex = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'

# Kiểm tra email hợp lệ
df['is_valid_email'] = df['email'].astype(str).str.match(email_regex)

# Đánh dấu các email không hợp lệ
print("Danh sách email không hợp lệ:")
df.loc[~df['is_valid_email'], ['order_id', 'customer_name_clean', 'email', 'is_valid_email']]


## 2. Phát Hiện & Xử Lý Giá Trị Ngoại Lai (Outliers)

### A. Kiểm Tra Logic Dữ Liệu (Domain Logic Constraints)
Trước khi dùng các thuật toán thống kê, hãy áp dụng logic nghiệp vụ:
- **Tuổi khách hàng**: Phải thuộc khoảng hợp lý `[10, 100]`. Các giá trị như `-5` hoặc `250` là lỗi nhập liệu chắc chắn.
- **Số lượng mua (`quantity`)**: Phải lớn hơn 0 và nhỏ hơn ngưỡng hợp lý (ví dụ: `<= 50` cho đơn hàng cá nhân).


In [ ]:
# Ép kiểu số
df['age_num'] = pd.to_numeric(df['age'], errors='coerce')
df['quantity_num'] = pd.to_numeric(df['quantity'], errors='coerce')

# Phát hiện lỗi logic
invalid_age = df[(df['age_num'] < 10) | (df['age_num'] > 100)]
invalid_qty = df[(df['quantity_num'] <= 0) | (df['quantity_num'] > 100)]

print("Dòng có tuổi không hợp lệ:")
print(invalid_age[['order_id', 'customer_name_clean', 'age_num']])

print("
Dòng có số lượng mua không hợp lệ:")
print(invalid_qty[['order_id', 'customer_name_clean', 'quantity_num']])


### B. Phát Hiện Outlier Bằng Phương Pháp IQR (Interquartile Range)
Công thức tính biên ngoại lai:
- $IQR = Q3 - Q1$
- $	ext{Lower Bound} = Q1 - 1.5 	imes IQR$
- $	ext{Upper Bound} = Q3 + 1.5 	imes IQR$


In [ ]:
# Tạo hàm phát hiện IQR Outliers
def detect_iqr_outliers(series):
    clean_series = series.dropna()
    q1 = clean_series.quantile(0.25)
    q3 = clean_series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return lower_bound, upper_bound

# Áp dụng cho cột quantity (lọc các giá trị dương)
valid_qty_series = df.loc[df['quantity_num'] > 0, 'quantity_num']
lb, ub = detect_iqr_outliers(valid_qty_series)
print(f"IQR Bounds cho Quantity: [{lb:.2f}, {ub:.2f}]")

qty_outliers = df[(df['quantity_num'] < lb) | (df['quantity_num'] > ub)]
print(f"Số lượng outliers được phát hiện: {len(qty_outliers)}")
qty_outliers[['order_id', 'customer_name_clean', 'quantity_num']]


### C. Xử Lý Outlier: Capping / Winsorizing (Giới Hạn Biên Với `.clip()`)
Thay vì xóa dòng làm mất dữ liệu của các cột khác, chúng ta có thể giới hạn các giá trị ngoại lai về cận trên / cận dưới hợp lý.


In [ ]:
# Giới hạn số lượng mua trong khoảng hợp lý [1, 10]
df['quantity_capped'] = df['quantity_num'].clip(lower=1, upper=10)

# Chuyển các tuổi không hợp lệ thành NaN và điền bằng median tuổi hợp lệ
median_age = df.loc[(df['age_num'] >= 10) & (df['age_num'] <= 100), 'age_num'].median()
df['age_cleaned'] = df['age_num'].apply(lambda x: x if (10 <= x <= 100) else median_age)

df[['order_id', 'age_num', 'age_cleaned', 'quantity_num', 'quantity_capped']].head(10)


## 3. Tổng Kết Xử Lý Text & Outliers
1. **Chuỗi văn bản**: Luôn dùng `.str.strip()`, chuẩn hóa hoa/thường và áp dụng bảng ánh xạ (dictionary) cho các cột phân loại lộn xộn.
2. **Email / SĐT**: Dùng biểu thức chính quy (Regex) để xác thực tính hợp lệ.
3. **Outliers**:
   - Ưu tiên logic nghiệp vụ (ví dụ: tuổi từ 10-100, số lượng > 0).
   - Dùng **IQR** cho dữ liệu lệch (skewed data) và **Z-score** cho dữ liệu phân phối chuẩn.
   - Chọn giải pháp xử lý: **Capping** (giữ lại dữ liệu) hoặc **Lọc bỏ/Impute** tùy theo bài toán cụ thể.
